# 第3章 演習（正解）: Tool呼び出しとMemory (Checkpointer) の実装

---

In [ ]:
%pip install -qU langchain langchain-openai langgraph pydantic

In [ ]:
import os
from google.colab import userdata

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "chap03-solution"
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("環境変数の設定が完了しました")

## 課題1の正解: カスタムツールを定義する

In [ ]:
# ✅ 正解
from langchain.tools import tool

@tool
def get_weather(city: str) -> str:
    """指定した都市の現在の天気情報を取得します。
    
    Args:
        city: 天気を調べたい都市名（例: 東京、大阪、名古屋）
    """
    weather_data = {
        "東京": "晴れ、気温20°C、湿度50%",
        "大阪": "曇り、気温18°C、湿度60%",
        "名古屋": "雨、気温15°C、湿度80%",
    }
    return weather_data.get(city, f"{city}の天気情報は取得できませんでした")


@tool
def calculate(expression: str) -> str:
    """数学の計算式を計算して結果を返します。
    
    Args:
        expression: 計算したい数式（例: '100 + 200 * 3'、'(50 - 10) / 4'）
    """
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"計算結果: {expression} = {result}"
    except Exception as e:
        return f"計算エラー: {str(e)}"


print(get_weather.invoke({"city": "東京"}))
print(calculate.invoke({"expression": "(100 + 200) * 3"}))

## 課題2の正解: ツール付きエージェントを作成する

In [ ]:
# ✅ 正解
from langchain.agents import create_agent

agent_with_tools = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather, calculate],   # ツールをリストで渡す
    system_prompt=(
        "あなたは親切なAIアシスタントです。\n"
        "天気の質問には get_weather ツールを、\n"
        "計算が必要な場合は calculate ツールを使ってください。\n"
        "日本語で回答してください。"
    ),
)

result = agent_with_tools.invoke({
    "messages": [
        {"role": "user", "content": "東京の天気を教えてください。また、123 × 456 はいくつですか？"}
    ]
})
print(result["messages"][-1].content)

## 課題3の正解: Checkpointer（Memory）を追加する

In [ ]:
# ✅ 正解
from langgraph.checkpoint.memory import InMemorySaver

agent_with_memory = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather, calculate],
    system_prompt="あなたは親切なAIアシスタントです。日本語で回答してください。",
    checkpointer=InMemorySaver(),   # ← InMemorySaver を渡す
)

# thread_id で会話セッションを管理
config = {"configurable": {"thread_id": "user_001"}}

result1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "東京の天気を教えてください"}]},
    config=config,
)
print("[ターン1]", result1["messages"][-1].content)

result2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "さっき調べた都市の気温は何度でしたか？"}]},
    config=config,   # 同じ thread_id → 会話履歴を引き継ぐ
)
print("[ターン2]", result2["messages"][-1].content)

## チャレンジ問題の正解: Structured Output

In [ ]:
# ✅ 正解
from pydantic import BaseModel, Field

class WeatherReport(BaseModel):
    city: str = Field(description="都市名")
    condition: str = Field(description="天気の状態（晴れ/曇り/雨など）")
    temperature_celsius: float = Field(description="気温（摂氏）")
    recommendation: str = Field(description="天気に基づく行動アドバイス")


structured_agent = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather],
    response_format=WeatherReport,   # ← 出力スキーマとして Pydantic モデルを指定
    system_prompt="天気ツールを使って情報を取得し、指定のフォーマットで回答してください。",
)

result = structured_agent.invoke({
    "messages": [{"role": "user", "content": "大阪の天気を調べて、外出するべきか教えてください"}]
})

report = result.get("structured_response")
if report:
    print(f"都市: {report.city}")
    print(f"天気: {report.condition}")
    print(f"気温: {report.temperature_celsius}°C")
    print(f"おすすめ: {report.recommendation}")